</br>
<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>Práctica 1:
        </h2>
        <h1 style="margin-top: -10px;">
            Análisis, procesamiento y validación
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Minería de Datos, 2025-26
                    </p>
                </div>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Estudiantes
        </h3>
    </div>
    <ul style="list-style-type: none; padding-left: 28px; margin: 0;  font-size: 1.1em">
        <li>Estudiante 1: Alex Ortega Redondo</li>
        <li>Estudiante 2: Javier García Meneses</li>
    </ul>

</div>

## 1. Introducción

El objetivo de esta práctica es aplicar las técnicas de un flujo de trabajo de **minería de datos** para resolver un problema de **clasificación supervisada**. Se analizará un conjunto de datos sobre el rendimiento y la actividad de estudiantes universitarios para predecir la probabilidad de abandono de sus estudios.

El desarrollo abarcará desde el **análisis exploratorio** de los datos y su **preprocesamiento**, hasta el **entrenamiento** de un modelo y su **validación** robusta. 

El rendimiento de los modelos se evaluará en una competición privada en Kaggle, que abarcará esta Práctica 1 y la Práctica 2 (selección de modelos).

<div style="border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: 10px">

## 2. Carga y visualización de los datos

Se proporciona un conjunto de datos sintético que simula el rendimiento académico y la interacción de estudiantes durante su primer año. Los datos contienen una variedad de tipos de variables, valores faltantes y otros artefactos que deberán ser gestionados.

Comenzamos importando las librerías necesarias y cargando los conjuntos de datos de entrenamiento (`train`) y de prueba (`test`) para poder examinarlos:

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

df_train = pd.read_csv('uclm_student_train.csv')

In [3]:
df_test = pd.read_csv('uclm_student_test.csv')

Las variables disponibles son:
`id`, `nombre`, `nacimiento`, `provincia`, `residencia_id`, `trabaja`, `horas_trabajo`, `bachillerato`, `nota_acceso`, `modalidad`, `creditos_a1`, `superados_a1`, `nota_s1`, `satisfaccion`, `horas_moodle`, `posts_foro`, `uso_biblioteca`, `eventos`, `tutorias`, `comentarios`, `numero_fav`, `talla_zapato`, `color_fav`, `meses_matriculado`, `grupo_trabajo`, `abandono`.

La variable objetivo es `abandono`, que es binaria:
* **1**: El estudiante abandona los estudios.
* **0**: El estudiante continúa.

Se puede observar cómo hay variables categóricas, numéricas, fechas, texto, valores faltantes... Esto requerirá de un análisis exploratorio y un preprocesamiento.

<div style="border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: 10px">

## 3. Flujo de Trabajo y Requisitos

### 3.1 Análisis Exploratorio de los Datos.

**Realizado en la otra libreta.**

### 3.2. Preprocesamiento de Datos
<img src="https://upload.wikimedia.org/wikipedia/commons/b/b9/CRISP-DM_Process_Diagram.png" alt="Diagrama de Proceso CRISP-DM" width="550"/>

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import GroupKFold

from utils import *  

Durante las pruebas, probamos a entrenar los modelos con la clasificación de comentarios que hicimos en el EDA, pero en la asignatura de Visión Artificial y Reconocimiento de Patrones nos explicaron un método para tratar los textos, **TF-iDF**. Este método consiste en que, tenemos un diccionario que nos indica las veces que ha aparecido una palabra en un texto, y si no ha aparecido ninguna vez, se le pone un 0. Tras comprobar todos los textos, si una palabra ha aparecido muchas veces en todos los textos, le resta importancia.<br>
Probamos a hacer una subida con este método y vimos que nos subió el score algo más que de la otra forma, por lo tanto nos hemos quedado de momento con **TF-iDF**.

Para el preprocesamiento de las variables, a las variables categóricas les vamos a aplicar **OneHotEncoder**, a las numéricas **StandardScaler**, a las binarias no les vamos a hacer nada, y a `comentarios` le vamos a aplicar **TF-iDF**.

Para la limpieza de las variables y la creación de las nuevas, lo hemos hecho todo en un utils con clases, y dentro del Pipeline vamos a llamar a esas clases para que realice el transforms en los datos.

#### 3.2.1 Árboles de Decisión

In [5]:
cat_vars = ['bachillerato', 'modalidad', 'prov_estudia', 'genero','domicilio','convocatoria','edificio_prov_estudia']
num_vars = ['nota_acceso','creditos_a1', 'nota_s1', 'satisfaccion','ratio_creditos_superados','participacion','horas_moodle','tamaño_grupo', 'horas_trabajo','edad_entrada'] 
binarias = ['horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0']

In [6]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()), 
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,       
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_vars),
    ('num', StandardScaler(), num_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

DT_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('participacion', Participacion()),
    ('convocatoria', Convocatoria()),
    ('nuevas_variables', NuevasVariables()),
    ('gp_tbj', GrupoTrabajo()),
    ('preproc', preproc),
    ("clf", DecisionTreeClassifier(random_state=1, class_weight='balanced', max_depth=10))
])

#### 3.2.2 Naive-Bayes Multinomial

In [7]:
cat_vars = ['horas_trabajo', 'bachillerato', 'modalidad', 'prov_estudia', 'genero', 'domicilio', 'nota_acceso_disc', 'nota_s1_disc', 'categoria_satisfaccion','nivel_participacion','tamaño_grupo','edificio_prov_estudia']
binarias = ['es_local','horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0', 'edad_entrada_bin']

In [8]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()),  
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,        
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

NBMultinomial_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('participacion', Participacion()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('convocatoria', Convocatoria()),
    ('nuevas_variables', NuevasVariables()),
    ('gp_tbj', GrupoTrabajo()),
    ('disc',Discretizar()),
    ('preproc', preproc),
    ("clf", MultinomialNB())
])

#### 3.2.3 Naive-Bayes Bernoulli

In [9]:
cat_vars = ['horas_trabajo', 'bachillerato', 'modalidad', 'prov_estudia', 'genero', 'domicilio', 'nota_acceso_disc', 'nota_s1_disc', 'categoria_satisfaccion','nivel_participacion','tamaño_grupo','edificio_prov_estudia']
binarias = ['es_local','horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0', 'edad_entrada_bin']

In [10]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()),  
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,        
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

NBBernoulli_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('participacion', Participacion()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('convocatoria', Convocatoria()),
    ('nuevas_variables', NuevasVariables()),
    ('gp_tbj', GrupoTrabajo()),
    ('disc',Discretizar()),
    ('preproc', preproc),
    ("clf", BernoulliNB())
])

### 3.3. Modelización y Validación


Para la validación de nuestros modelos, vamos a utilizar **GroupKFold**, ya que tenemos una variable llamada **tamaño_grupo** extraída de grupo_trabajo, la cual nos genera problemas en las particiones de los datos de entrenamientos en train y validación.

**Un ejemplo claro sería**: Tenemos un grupo de trabajo con 10 miembros y en las particiones se quedan 6 en train y 4 en la validación, con lo cual estamos perdiendo información real. Para eso usamos esta validación que lo que hace es que ningún grupo se reparte entre train y validación en el mismo fold.

Hemos probado a usar diferentes validacion como **StratifiedKFold**, pero al tener esta variable, la partición separaba los miembros de un mismo grupo y nos empeoraba el score en local.

In [11]:
df_train = pd.read_csv('uclm_student_train.csv')

In [12]:
X = df_train.drop(columns=["abandono"])
y = df_train["abandono"]

#### 3.3.1 Árboles de Decisión

In [13]:
group_kfold = GroupKFold(n_splits=5)
groups = X['grupo_trabajo']

scoring = ['accuracy', 'f1', 'precision', 'recall']
scores = cross_validate(DT_pipeline,X, y,cv=group_kfold,groups=groups,scoring=['f1', 'accuracy', 'precision', 'recall'],)
print("Cross-validated scores: \n")
for metric in scoring:
    print(f"{metric}: {scores['test_' + metric].mean():.4f} ± {scores['test_' + metric].std():.4f}")

Cross-validated scores: 

accuracy: 0.8376 ± 0.0045
f1: 0.6502 ± 0.0048
precision: 0.7311 ± 0.0041
recall: 0.5854 ± 0.0074


#### 3.3.2 Naive-Bayes Multinomial

In [14]:
group_kfold = GroupKFold(n_splits=5)
groups = X['grupo_trabajo']

scoring = ['accuracy', 'f1', 'precision', 'recall']
scores = cross_validate(NBMultinomial_pipeline,X, y,cv=group_kfold,groups=groups,scoring=['f1', 'accuracy', 'precision', 'recall'],)
print("Cross-validated scores: \n")
for metric in scoring:
    print(f"{metric}: {scores['test_' + metric].mean():.4f} ± {scores['test_' + metric].std():.4f}")

Cross-validated scores: 

accuracy: 0.8595 ± 0.0025
f1: 0.6798 ± 0.0041
precision: 0.8238 ± 0.0085
recall: 0.5787 ± 0.0052


#### 3.3.3 Naive-Bayes Bernoulli

In [15]:
group_kfold = GroupKFold(n_splits=5)
groups = X['grupo_trabajo']

scoring = ['accuracy', 'f1', 'precision', 'recall']
scores = cross_validate(NBBernoulli_pipeline,X, y,cv=group_kfold,groups=groups,scoring=['f1', 'accuracy', 'precision', 'recall'],)
print("Cross-validated scores: \n")
for metric in scoring:
    print(f"{metric}: {scores['test_' + metric].mean():.4f} ± {scores['test_' + metric].std():.4f}")

Cross-validated scores: 

accuracy: 0.8403 ± 0.0016
f1: 0.6759 ± 0.0074
precision: 0.7083 ± 0.0116
recall: 0.6464 ± 0.0056


### 3.4. Predicción y Evaluación

In [16]:
df_train = pd.read_csv('uclm_student_train.csv')

In [17]:
X_train = df_train.drop(columns=["abandono"])
y_train = df_train["abandono"]

In [18]:
X_test = pd.read_csv('uclm_student_test.csv')

In [19]:
id_test = X_test[['id']].copy()

#### 3.4.1 Árboles de Decisión

In [20]:
cat_vars = ['bachillerato', 'modalidad', 'prov_estudia', 'genero','domicilio','convocatoria','edificio_prov_estudia']
num_vars = ['nota_acceso','creditos_a1', 'nota_s1', 'satisfaccion','ratio_creditos_superados','participacion','horas_moodle','tamaño_grupo', 'horas_trabajo','edad_entrada'] 
binarias = ['horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0']

In [21]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()), 
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,       
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_vars),
    ('num', StandardScaler(), num_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

DT_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('participacion', Participacion()),
    ('convocatoria', Convocatoria()),
    ('grupo_trabajo', GrupoTrabajo()),
    ('nuevas_variables', NuevasVariables()),
    ('preproc', preproc),
    ("clf", DecisionTreeClassifier(random_state=1, class_weight='balanced', max_depth=10))
])

In [22]:
DT_pipeline.fit(X_train, y_train)
y_pred = DT_pipeline.predict(X_test)

id_test['abandono'] = y_pred
id_test[['id','abandono']].to_csv('Prueba1.csv',index=False)

#### 3.4.2 Naive-Bayes Multinomial

In [23]:
cat_vars = ['horas_trabajo', 'bachillerato', 'modalidad', 'prov_estudia', 'genero', 'domicilio', 'nota_acceso_disc', 'nota_s1_disc', 'categoria_satisfaccion','nivel_participacion','tamaño_grupo','edificio_prov_estudia']
binarias = ['es_local','horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0', 'edad_entrada_bin']

In [24]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()),  
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,        
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

NBMultinomial_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('participacion', Participacion()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('convocatoria', Convocatoria()),
    ('nuevas_variables', NuevasVariables()),
    ('gp_tbj', GrupoTrabajo()),
    ('disc',Discretizar()),
    ('preproc', preproc),
    ("clf", MultinomialNB())
])

In [25]:
NBMultinomial_pipeline.fit(X_train, y_train)
y_pred = NBMultinomial_pipeline.predict(X_test)

id_test['abandono'] = y_pred
id_test[['id','abandono']].to_csv('Prueba2.csv',index=False)

#### 3.4.3 Naive-Bayes Bernoulli

In [26]:
cat_vars = ['horas_trabajo', 'bachillerato', 'modalidad', 'prov_estudia', 'genero', 'domicilio', 'nota_acceso_disc', 'nota_s1_disc', 'categoria_satisfaccion','nivel_participacion','tamaño_grupo','edificio_prov_estudia']
binarias = ['es_local','horas_trabajo_missing', 'nota_s1_missing','mes_matriculado','superados_0', 'edad_entrada_bin']

In [27]:
text_pipeline = Pipeline([
    ('extract', ExtractComentario()),  
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents='unicode',
        max_df=0.9,        
        min_df=5,          
        ngram_range=(1,2)  
    ))
])

preproc = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_vars),
    ('bin', 'passthrough', binarias),
    ('txt', text_pipeline, ['comentarios'])
])

NBBernoulli_pipeline = Pipeline([
    ('limpieza', LimpiezaVariables()),
    ('participacion', Participacion()),
    ('imputacion_horas', ImputarHorasTrabajo()),
    ('localidad', Localidad()),
    ('genero', Genero()),
    ('edad_entrada', EdadEntrada()),
    ('ratio_creditos', RatioCredito()),
    ('imputacion_nota', ImputarNota()),
    ('convocatoria', Convocatoria()),
    ('nuevas_variables', NuevasVariables()),
    ('gp_tbj', GrupoTrabajo()),
    ('disc',Discretizar()),
    ('preproc', preproc),
    ("clf", BernoulliNB())
])

In [28]:
NBBernoulli_pipeline.fit(X_train, y_train)
y_pred = NBBernoulli_pipeline.predict(X_test)

id_test['abandono'] = y_pred
id_test[['id','abandono']].to_csv('Prueba3.csv',index=False)

### 4. Conclusión

Durante esta práctica hemos aprendido mucho sobre el **proceso completo de minería de datos**, desde la limpieza inicial hasta la creación de modelos predictivos. Empezamos corrigiendo algunos datos que venían en formatos incorrectos o consistentes, para poder trabajar con un **conjunto limpio y coherente**.

Después dedicamos la mayor parte del tiempo al análisis exploratorio **(EDA)**, revisando cada variable por separado y buscando relaciones interesantes entre ellas. A medida que analizábamos, se nos fueron ocurriendo nuevas **variables derivadas** que ayudaban a explicar mejor el abandono, así que también las fuimos creando y probando.

Una vez teníamos una visión más clara del dataset, utilizamos un **árbol de clasificación** para guiarnos en la selección de las variables más relevantes. A partir de ahí, empezamos a montar distintos **pipelines**, uno por cada modelo que queríamos probar **(árboles y Naive Bayes)**.

El trabajo fue bastante **iterativo**: probábamos un modelo, mirábamos los resultados, cambiábamos alguna variable o ajustábamos algo del preprocesado, y volvíamos a probar. Gracias a eso, conseguimos entender mejor cómo afectaban las transformaciones y qué tipo de variables funcionaban mejor para el problema.

En general, esta práctica nos ha servido para ver lo importante que es analizar bien los datos antes de modelar, entender qué aporta cada variable y experimentar poco a poco hasta conseguir un buen resultado. Además, nos quedamos con la idea de que, aunque los modelos que usamos eran simples, **si se trabajan bien pueden dar resultados muy buenos**.